In [2]:
import altair as alt
import numpy as np
import pandas as pd

Loading the results dataset

In [3]:
results_data = pd.read_csv("results.csv")
results_data

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False
...,...,...,...,...,...,...,...,...,...
49066,2026-01-18,Bolivia,Panama,1,1,Friendly,Tarija,Bolivia,False
49067,2026-01-18,Grenada,Jamaica,0,1,Friendly,St. George's,Grenada,False
49068,2026-01-22,Panama,Mexico,0,1,Friendly,Panama City,Panama,False
49069,2026-01-25,Bolivia,Mexico,0,1,Friendly,Santa Cruz,Bolivia,False


Sorting the results data to only consider obervations(matches) from 2015 onwards

In [4]:
# Only considering matches starting after 2015
results_data['date'] = pd.to_datetime(results_data['date'])
results_filtered = results_data[results_data['date'] >= '2015-01-01']
results_filtered

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
38380,2015-01-04,Bahrain,Jordan,1,0,Friendly,Ballarat,Australia,True
38381,2015-01-04,Iran,Iraq,1,0,Friendly,Wollongong,Australia,True
38382,2015-01-04,South Korea,Saudi Arabia,2,0,Friendly,Parramatta,Australia,True
38383,2015-01-04,South Africa,Zambia,1,0,Friendly,Johannesburg,South Africa,False
38384,2015-01-05,China PR,Oman,4,1,Friendly,Penrith,Australia,True
...,...,...,...,...,...,...,...,...,...
49066,2026-01-18,Bolivia,Panama,1,1,Friendly,Tarija,Bolivia,False
49067,2026-01-18,Grenada,Jamaica,0,1,Friendly,St. George's,Grenada,False
49068,2026-01-22,Panama,Mexico,0,1,Friendly,Panama City,Panama,False
49069,2026-01-25,Bolivia,Mexico,0,1,Friendly,Santa Cruz,Bolivia,False


Loading the rankings dataset and Converting the rank_date variable into a 'date' for python

In [5]:
rankings_data = pd.read_csv("fifa_ranking-2024-06-20.csv")
rankings_data["year"] = pd.to_datetime(rankings_data["rank_date"]).dt.year
rankings_data


,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date,year
0,140.0,Brunei Darussalam,BRU,2.00,0.00,140,AFC,1992-12-31,1992
1,33.0,Portugal,POR,38.00,0.00,33,UEFA,1992-12-31,1992
2,32.0,Zambia,ZAM,38.00,0.00,32,CAF,1992-12-31,1992
3,31.0,Greece,GRE,38.00,0.00,31,UEFA,1992-12-31,1992
4,30.0,Algeria,ALG,39.00,0.00,30,CAF,1992-12-31,1992
...,...,...,...,...,...,...,...,...,...
67467,137.0,Kuwait,KUW,1098.42,1085.46,-2,AFC,2024-06-20,2024
67468,136.0,Lithuania,LTU,1100.66,1095.23,-1,UEFA,2024-06-20,2024
67469,135.0,Malaysia,MAS,1107.58,1094.54,-3,AFC,2024-06-20,2024
67470,133.0,Solomon Islands,SOL,1111.02,1111.02,1,OFC,2024-06-20,2024


Filter the rankings data to only consider rankings from 2015 onwards and for each country to only have 1 rankings per year

In [6]:
# Only considering rankings starting after 2015 and only have 1 observation per year per team
rankings_filtered = (
    rankings_data[rankings_data['year'] >= 2015]
    .groupby(['country_full', 'year'])
    .last()
    .reset_index()
)
rankings_filtered

,country_full,year,rank,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
0,Afghanistan,2015,150.0,AFG,194.00,168.00,-6,AFC,2015-12-03
1,Afghanistan,2016,146.0,AFG,189.00,189.00,-1,AFC,2016-12-22
2,Afghanistan,2017,148.0,AFG,181.00,181.00,1,AFC,2017-12-21
3,Afghanistan,2018,147.0,AFG,1068.00,1068.00,0,AFC,2018-12-20
4,Afghanistan,2019,149.0,AFG,1052.00,1052.00,0,AFC,2019-12-19
...,...,...,...,...,...,...,...,...,...
2101,Zimbabwe,2020,108.0,ZIM,1181.00,1181.00,0,CAF,2020-12-10
2102,Zimbabwe,2021,121.0,ZIM,1138.44,1138.44,0,CAF,2021-12-23
2103,Zimbabwe,2022,125.0,ZIM,1138.56,1138.56,0,CAF,2022-12-22
2104,Zimbabwe,2023,124.0,ZIM,1144.56,1144.56,0,CAF,2023-12-21


Some countries are spelled differently in the datasets, so this makes them the same

In [7]:
name_mapping = {
    'USA':            'United States',
    'Korea Republic': 'South Korea',
    'Congo DR':       'DR Congo',
    "Cote d'Ivoire":  'Ivory Coast',
    'IR Iran':        'Iran',
}
rankings_data['country_full'] = rankings_data['country_full'].replace(name_mapping)
rankings_filtered['country_full'] = rankings_filtered['country_full'].replace(name_mapping)

Teams that have already qualified for the 2026 world cup

In [8]:
world_cup_2026_teams = [
    # Host nations (auto-qualified)
    "United States", "Canada", "Mexico",
    # UEFA
    "Germany", "Portugal", "Spain", "France", "England", "Netherlands",
    "Belgium", "Austria", "Turkey", "Poland", "Serbia",
    # CONMEBOL
    "Argentina", "Colombia", "Uruguay", "Brazil", "Ecuador", "Paraguay",
    # CONCACAF
    "Panama", "Costa Rica", "Honduras", "Jamaica",
    # CAF
    "Morocco", "Egypt", "Senegal", "South Africa", "DR Congo",
    "Ivory Coast", "Nigeria", "Algeria", "Tunisia",
    # AFC
    "Iran", "South Korea", "Japan", "Saudi Arabia", "Australia",
    "Uzbekistan", "Jordan", "Iraq",
    # OFC
    "New Zealand",
]

Filter to only include matches playd between two teams that have qualified for the 2026 world cup

In [9]:
# Only includes teams currently qualified for the wolrd cup
results_wc2026 = results_filtered[
    results_filtered['home_team'].isin(world_cup_2026_teams) &
    results_filtered['away_team'].isin(world_cup_2026_teams)
]
results_wc2026

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
38381,2015-01-04,Iran,Iraq,1,0,Friendly,Wollongong,Australia,True
38382,2015-01-04,South Korea,Saudi Arabia,2,0,Friendly,Parramatta,Australia,True
38395,2015-01-11,Nigeria,Ivory Coast,0,1,Friendly,Abu Dhabi,United Arab Emirates,True
38396,2015-01-11,Tunisia,Algeria,1,1,Friendly,Radès,Tunisia,False
38399,2015-01-12,Jordan,Iraq,0,1,AFC Asian Cup,Brisbane,Australia,True
...,...,...,...,...,...,...,...,...,...
49062,2026-01-14,Senegal,Egypt,1,0,African Cup of Nations,Tangier,Morocco,True
49063,2026-01-14,Morocco,Nigeria,0,0,African Cup of Nations,Rabat,Morocco,False
49064,2026-01-17,Egypt,Nigeria,0,0,African Cup of Nations,Casablanca,Morocco,True
49065,2026-01-18,Morocco,Senegal,0,1,African Cup of Nations,Rabat,Morocco,False


Filter to only include rankings for teams that are qualified for the 2026 world cup

In [10]:
# Only includes teams currently qualified for the wolrd cup
rankings_wc2026 = rankings_filtered[
    rankings_filtered['country_full'].isin(world_cup_2026_teams)
].sort_values(by=['rank_date', 'rank'], ascending=[False, True])
rankings_wc2026

,country_full,year,rank,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
89,Argentina,2024,1.0,ARG,1860.14,1858.00,0,CONMEBOL,2024-06-20
717,France,2024,2.0,FRA,1837.47,1840.59,0,UEFA,2024-06-20
199,Belgium,2024,3.0,BEL,1797.98,1795.23,0,UEFA,2024-06-20
279,Brazil,2024,4.0,BRA,1791.85,1788.65,-1,CONMEBOL,2024-06-20
627,England,2024,5.0,ENG,1787.88,1794.90,1,UEFA,2024-06-20
...,...,...,...,...,...,...,...,...,...
967,Jordan,2015,87.0,JOR,399.00,411.00,5,AFC,2015-12-03
360,Canada,2015,88.0,CAN,388.00,335.00,-14,CONCACAF,2015-12-03
917,Iraq,2015,89.0,IRQ,381.00,392.00,2,AFC,2015-12-03
847,Honduras,2015,101.0,HON,338.00,359.00,6,CONCACAF,2015-12-03


Adding nations FIFA ranking for each match played

In [11]:
# So the 2 datasets can correlate the countries rank at the time of a match with each other
results_wc2026['year'] = results_wc2026['date'].dt.year
rankings_lookup = rankings_wc2026[['country_full', 'year', 'rank']]


results_wc2026 = (
    results_wc2026.merge(
    rankings_lookup.rename(columns={'country_full': 'home_team', 'rank': 'home_ranking'})
    ).merge(
    rankings_lookup.rename(columns={'country_full': 'away_team', 'rank': 'away_ranking'})
    )
)
results_wc2026 = results_wc2026[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'neutral', 'home_ranking', 'away_ranking']]
results_wc2026

/var/folders/pb/9jlgn7sn1qnc1ndptsdnjrs00000gn/T/ipykernel_10214/3908683382.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results_wc2026['year'] = results_wc2026['date'].dt.year


,date,home_team,away_team,home_score,away_score,tournament,neutral,home_ranking,away_ranking
0,2015-01-04,Iran,Iraq,1,0,Friendly,True,45.0,89.0
1,2015-01-04,South Korea,Saudi Arabia,2,0,Friendly,True,51.0,80.0
2,2015-01-11,Tunisia,Algeria,1,1,Friendly,False,40.0,28.0
3,2015-01-12,Jordan,Iraq,0,1,AFC Asian Cup,True,87.0,89.0
4,2015-01-16,Iraq,Japan,0,1,AFC Asian Cup,True,89.0,53.0
...,...,...,...,...,...,...,...,...,...
1018,2024-11-18,United States,Jamaica,4,2,CONCACAF Nations League,False,11.0,53.0
1019,2024-11-19,Mexico,Honduras,4,0,CONCACAF Nations League,False,15.0,78.0
1020,2024-11-19,Colombia,Ecuador,0,1,FIFA World Cup qualification,False,12.0,30.0
1021,2024-11-19,Brazil,Uruguay,1,1,FIFA World Cup qualification,False,4.0,14.0


### Functions that sorts the 'Tournament' column in the DataFrame

In [12]:
def get_tournament_types(t = 'tournament'):
    '''
    Returns the different types of values in the tournaments column
    '''
    return results_data[t].drop_duplicates().sort_values()

tournaments = get_tournament_types()

tournaments

34280                    ABCS Tournament
4317                       AFC Asian Cup
4213         AFC Asian Cup qualification
29880                  AFC Challenge Cup
31784    AFC Challenge Cup qualification
                      ...               
13117                   West African Cup
5204         Windward Islands Tournament
39936                    World Unity Cup
6144     Zambian Independence Tournament
163                 Évence Coppée Trophy
Name: tournament, Length: 191, dtype: object

### Displays how many times each unique tournament type is mentioned in the data under the tournaments column

In [13]:
results_wc2026['tournament'].value_counts()

tournament
Friendly                                380
FIFA World Cup qualification            228
FIFA World Cup                           77
Copa América                             63
Gold Cup                                 45
UEFA Nations League                      44
CONCACAF Nations League                  37
UEFA Euro                                29
African Cup of Nations                   28
AFC Asian Cup                            23
UEFA Euro qualification                  17
Kirin Challenge Cup                      11
Arab Cup                                  8
African Cup of Nations qualification      8
Confederations Cup                        6
EAFF Championship                         4
Superclásico de las Américas              3
FIFA Series                               3
Gulf Cup                                  2
WAFF Championship                         1
UNCAF Cup                                 1
COSAFA Cup                                1
Kirin Cup            

### Assigns a numerical weight to each tournament type (1 = most important, and 0 = meaningless)

I will not include the 'tournament' that only occured 2-3 times in our data.

We can manipulate these values in case our data in the end does not seem reasonable

In [ ]:
tournament_weights = {
    'FIFA World Cup': 1.00,
    'Copa América' : 0.85,
    'UEFA Euro' : 0.85,
    'AFC Asian Cup': 0.85,
    'African Cup of Nations': 0.85,
    'Gold Cup' : 0.80,
    'Confederations Cup': 0.75, # Lower because it was discontinuied in 2019 and the final tournamnet was in 2017
    'CONCACAF Nations League': 0.75,
    'UEFA Nations League': 0.75,
    'Superclásico de las Américas': 0.65,
    'FIFA World Cup qualification': 0.65,
    'EAFF Championship': 0.65,
    'Arab Cup': 0.60,
    'UEFA Euro qualification': 0.55,
    'Copa América qualification': 0.55,
    'WAFF Championship': 0.55,
    'FIFA Series': 0.55,
    'Gulf Cup': 0.55,
    'African Cup of Nations qualification': 0.50,
    'UNCAF Cup': 0.50,
    'COSAFA Cup': 0.50,
    'CAFA Nations Cup': 0.5,
    'Kirin Challenge Cup': 0.35,
    'Kirin Cup': 0.35,
    'Soccer Ashes': 0.30,
    'Friendly': 0.20,
}

results_wc2026_copy = results_wc2026.copy()                             #Copies filtered data 
results_wc2026_copy['tournament_weight'] = (                           #Creates a new column called 'tournament_weight'
    results_wc2026_copy['tournament'].map(tournament_weights).fillna(0.30) #Assigns 0.30 to any tournament that was not mentioned 
)

results_wc2026_copy[['tournament', 'tournament_weight']                     #displays the new columns,removes duplicates,                     
    ].drop_duplicates().sort_values('tournament_weight', ascending = False) # and sorts them from highest to lowest weight

,tournament,tournament_weight
340,FIFA World Cup,1.00
7,African Cup of Nations,0.85
46,Copa América,0.85
138,UEFA Euro,0.85
3,AFC Asian Cup,0.85
59,Gold Cup,0.80
230,Confederations Cup,0.75
503,CONCACAF Nations League,0.75
378,UEFA Nations League,0.75
70,EAFF Championship,0.65


### Creating a goal differential 

Goal differntial is alread a number, but a 7-0 blowout should not mean a 7x chance to win the World Cup. I will cap it at 3 to reduce the influence of lopsided scores

In [36]:
results_wc2026['goal_diff'] = results_wc2026['home_score'] - results_wc2026['away_score']

results_wc2026['goal_diff_capped'] = results_wc2026['goal_diff'].clip(-3,3)

results_wc2026[['home_team', 'away_team', 'home_score', 'away_score', 'goal_diff', 'goal_diff_capped']].head(10)

,home_team,away_team,home_score,away_score,goal_diff,goal_diff_capped
0,Iran,Iraq,1,0,1,1
1,South Korea,Saudi Arabia,2,0,2,2
2,Tunisia,Algeria,1,1,0,0
3,Jordan,Iraq,0,1,-1,-1
4,Iraq,Japan,0,1,-1,-1
5,Australia,South Korea,0,1,-1,-1
6,Uzbekistan,Saudi Arabia,3,1,2,2
7,Algeria,South Africa,3,1,2,2
8,Japan,Jordan,2,0,2,2
9,South Korea,Uzbekistan,2,0,2,2


### Creating ranking differential

A positive ranking difference means that the home team is a higher rank compared to the away team. This will let us detect upsets

In [20]:
results_wc2026['ranking_diff'] = results_wc2026['away_ranking'] - results_wc2026['home_ranking']

results_wc2026[['home_team', 'home_ranking', 'away_team', 'away_ranking', 'ranking_diff']].head(10)

,home_team,home_ranking,away_team,away_ranking,ranking_diff
0,Iran,45.0,Iraq,89.0,44.0
1,South Korea,51.0,Saudi Arabia,80.0,29.0
2,Tunisia,40.0,Algeria,28.0,-12.0
3,Jordan,87.0,Iraq,89.0,2.0
4,Iraq,89.0,Japan,53.0,-36.0
5,Australia,57.0,South Korea,51.0,-6.0
6,Uzbekistan,74.0,Saudi Arabia,80.0,6.0
7,Algeria,28.0,South Africa,72.0,44.0
8,Japan,53.0,Jordan,87.0,34.0
9,South Korea,51.0,Uzbekistan,74.0,23.0


### Home Advantage 

Checks the netural column to detect if there is a home advantage or not.
1 = played at home ground, 0 = neutral venue

In [24]:
results_wc2026['home_advantage'] = (~results_wc2026['neutral']).astype(int) 

results_wc2026[['home_team', 'away_team', 'neutral', 'home_advantage']].head(10)

,home_team,away_team,neutral,home_advantage
0,Iran,Iraq,True,0
1,South Korea,Saudi Arabia,True,0
2,Tunisia,Algeria,False,1
3,Jordan,Iraq,True,0
4,Iraq,Japan,True,0
5,Australia,South Korea,False,1
6,Uzbekistan,Saudi Arabia,True,0
7,Algeria,South Africa,True,0
8,Japan,Jordan,True,0
9,South Korea,Uzbekistan,True,0


### Receny Weight

Makes it so the recent matches will weigh more

In [31]:
ref_date = results_wc2026['date'].max()
results_wc2026['days_ago'] = (ref_date - results_wc2026['date']).dt.days

results_wc2026['recency_weight'] = np.exp(-results_wc2026['days_ago'] / 365)

results_wc2026[['date', 'days_ago', 'recency_weight']].sort_values('date', ascending = False).head(10)

,date,days_ago,recency_weight
1022,2024-12-28,0,1.000000
1021,2024-11-19,39,0.898661
1020,2024-11-19,39,0.898661
1019,2024-11-19,39,0.898661
1018,2024-11-18,40,0.896202
1017,2024-11-18,40,0.896202
1014,2024-11-15,43,0.888867
1015,2024-11-15,43,0.888867
1016,2024-11-15,43,0.888867
1013,2024-11-14,44,0.886435


### Updated DataFrame

In [32]:
results_wc2026

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_ranking,away_ranking,goal_diff,goal_diff_capped,ranking_diff,home_advantage,days_ago,recency_weight
0,2015-01-04,Iran,Iraq,1,0,Friendly,True,45.0,89.0,1,1,44.0,0,3646,0.000046
1,2015-01-04,South Korea,Saudi Arabia,2,0,Friendly,True,51.0,80.0,2,2,29.0,0,3646,0.000046
2,2015-01-11,Tunisia,Algeria,1,1,Friendly,False,40.0,28.0,0,0,-12.0,1,3639,0.000047
3,2015-01-12,Jordan,Iraq,0,1,AFC Asian Cup,True,87.0,89.0,-1,-1,2.0,0,3638,0.000047
4,2015-01-16,Iraq,Japan,0,1,AFC Asian Cup,True,89.0,53.0,-1,-1,-36.0,0,3634,0.000047
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1018,2024-11-18,United States,Jamaica,4,2,CONCACAF Nations League,False,11.0,53.0,2,2,42.0,1,40,0.896202
1019,2024-11-19,Mexico,Honduras,4,0,CONCACAF Nations League,False,15.0,78.0,4,3,63.0,1,39,0.898661
1020,2024-11-19,Colombia,Ecuador,0,1,FIFA World Cup qualification,False,12.0,30.0,-1,-1,18.0,1,39,0.898661
1021,2024-11-19,Brazil,Uruguay,1,1,FIFA World Cup qualification,False,4.0,14.0,0,0,10.0,1,39,0.898661
